# Characterisation Step (PMDCo / OBI): from data entry to RDF

This notebook shows how to describe a generic characterisation step and
convert it into a standardised, machine-readable RDF graph following the
[OBI](http://purl.obolibrary.org/obo/obi.owl) and
[PMDCo](https://w3id.org/pmd/co/) ontologies.

**You only need to edit one file:** `docs/example.input.json`. Everything
else is automatic.

This is the **generic base schema** for characterisation. It covers any
measurement or experimental test on a material specimen. For specific test
types with richer vocabulary (e.g. tensile testing with yield strength,
tensile strength, elongation), use the specialised schemas:

- `characterization/tensile-test/TTO/` for ISO 6892-1 tensile tests

A characterisation step captures:

- a **name** and optional **description**
- the **specimen or material** it tests (by IRI)
- which **manufacturing step** produced the specimen being tested
- optional **test conditions** (standard, strain rate, temperature, etc.)


## What the notebook does

```
example.input.json
  |  your test name, specimen IRI, and test conditions
  |
  v
Transform
  |  converts your plain JSON into a structured OO-LD document
  |
  v
RDF graph
  |  machine-readable, ontology-aligned triples
  |
  v
SHACL validation  ->  confirms the graph is structurally correct
SPARQL inspect    ->  shows the key connections recorded in the graph
```


## Environment setup

```bash
python3 -m venv .venv
source .venv/bin/activate
pip install jupyterlab
jupyter lab
```

In [1]:
%pip install -q jsonata-python rdflib pyshacl pyyaml

Note: you may need to restart the kernel to use updated packages.


In [2]:
import sys, json, pathlib, yaml, pyshacl, rdflib
from jsonata.jsonata import Jsonata
from importlib.metadata import version

print("Python        :", sys.executable)
print("jsonata-python", version("jsonata-python"))
print("rdflib        ", version("rdflib"))
print("pyshacl       ", version("pyshacl"))

HERE   = pathlib.Path().resolve()   # docs/
SCHEMA = HERE.parent                # characterization/step/PMDCo/

Python        : /root/semantic-dataspace/.venv/bin/python3
jsonata-python 0.6.2
rdflib         7.6.0
pyshacl        0.31.0


## Step 1: Describe your characterisation

Edit `docs/example.input.json` with your data, then run this cell to load it.

| Field | Required | Description |
|---|---|---|
| `step_name` | yes | A name for this characterisation step |
| `description` | no | Free-text explanation of what was tested and how |
| `inputs` | no | List of IRIs of specimens or materials tested |
| `preceded_by` | no | List of IRIs of steps that directly precede this one |
| `conditions` | no | List of test conditions (standard, strain rate, temperature, ...) |
| `step_id` | no | Custom IRI slug; auto-derived from `step_name` if omitted |

**What is an IRI?**  
An IRI (Internationalized Resource Identifier) is a web address used to uniquely
identify a thing in a knowledge graph. For specimen inputs, use the IRI that
was assigned when the specimen was registered. For example:
`https://example.org/specimens/316L-tensile-bar-1`.  
If specimens do not yet have IRIs, leave `inputs` empty for now and add them later.

**When to use this schema vs. the tensile-test schema:**  
Use `characterization/step/PMDCo/` for any generic test or measurement where
the result vocabulary is not predefined. Use
`characterization/tensile-test/TTO/` when you have a full ISO 6892-1 tensile
test result with yield strength, tensile strength, elongation, and reduction of area.

In [3]:
simplified = json.loads((HERE / "example.input.json").read_text())

print(json.dumps(simplified, indent=2))

{
  "step_name": "Tensile test specimen 1",
  "description": "Uniaxial tensile test on a 316L stainless steel specimen at room temperature following ISO 6892-1.",
  "inputs": [
    "https://example.org/specimens/316L-tensile-bar-1"
  ],
  "conditions": [
    {
      "name": "Test Standard",
      "unit": "ISO 6892-1"
    },
    {
      "name": "Strain Rate",
      "value": 0.00025,
      "unit": "1/s"
    },
    {
      "name": "Temperature",
      "value": 23,
      "unit": "\u00b0C"
    }
  ]
}


## Step 2: Convert to OO-LD

This step transforms your plain JSON into a structured
[OO-LD](https://github.com/OO-LD/oold-python) document.
OO-LD is standard JSON enriched with ontology mappings: every field name
is linked to a precise ontology term, so machines can interpret it unambiguously.

The converter assigns stable identifiers to each node. By default, the step
identifier is derived from the step name (e.g. `"Tensile test specimen 1"` becomes
`char-step-tensile-test-specimen-1`). You can override this with `step_id`.

You can also run the transform from the command line:
```
python -m jsonata -e ../simplified/transform.jsonata -i example.input.json
```

In [4]:
expr     = (SCHEMA / "simplified" / "transform.jsonata").read_text()
oold_doc = Jsonata(expr).evaluate(simplified)

print(json.dumps(oold_doc, indent=2))

{
  "conforms_to": "https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/characterization/step/PMDCo/#v1.0.0",
  "type": "obi:0000070",
  "id": "char-step-tensile-test-specimen-1",
  "label": "Tensile test specimen 1",
  "description": "Uniaxial tensile test on a 316L stainless steel specimen at room temperature following ISO 6892-1.",
  "has_specified_input": [
    "https://example.org/specimens/316L-tensile-bar-1"
  ],
  "has_process_condition": [
    {
      "type": "pmdco:PMD_0000013",
      "id": "char-step-tensile-test-specimen-1-condition-1",
      "parameter_label": "Test Standard",
      "parameter_unit": "ISO 6892-1"
    },
    {
      "type": "pmdco:PMD_0000013",
      "id": "char-step-tensile-test-specimen-1-condition-2",
      "parameter_label": "Strain Rate",
      "parameter_value": 0.00025,
      "parameter_unit": "1/s"
    },
    {
      "type": "pmdco:PMD_0000013",
      "id": "char-step-tensile-test-specimen-1-condition-3",
      "parameter_label"

## Step 3: Convert to RDF

The OO-LD document is parsed into an RDF graph using the ontology context from
`specs/schema.oold.yaml`.

The characterisation step is typed as `obi:OBI_0000070` (Assay), the standard
OBI class for experimental measurements.

In [5]:
context = yaml.safe_load((SCHEMA / "specs" / "schema.oold.yaml").read_text())["@context"]

g = rdflib.Dataset()
g.parse(data=json.dumps({"@context": context, **oold_doc}), format="json-ld")

print(f"Graph contains {len(g)} triples.\n")
print(g.serialize(format="turtle"))

Graph contains 19 triples.

@prefix dcterms: <http://purl.org/dc/terms/> .
@prefix ns1: <http://purl.obolibrary.org/obo/> .
@prefix pmdco: <https://w3id.org/pmd/co/> .
@prefix qudt: <http://qudt.org/schema/qudt/> .
@prefix rdfs: <http://www.w3.org/2000/01/rdf-schema#> .
@prefix xsd: <http://www.w3.org/2001/XMLSchema#> .

<https://w3id.org/pmd/co/test/char-step-tensile-test-specimen-1> a ns1:OBI_0000070 ;
    rdfs:label "Tensile test specimen 1" ;
    ns1:OBI_0000293 <https://example.org/specimens/316L-tensile-bar-1> ;
    dcterms:conformsTo <https://github.com/semantic-dataspace/semantic-schemas/tree/main/schemas/characterization/step/PMDCo/#v1.0.0> ;
    rdfs:comment "Uniaxial tensile test on a 316L stainless steel specimen at room temperature following ISO 6892-1." ;
    pmdco:PMD_0000016 <https://w3id.org/pmd/co/test/char-step-tensile-test-specimen-1-condition-1>,
        <https://w3id.org/pmd/co/test/char-step-tensile-test-specimen-1-condition-2>,
        <https://w3id.org/pmd/co/t

In [6]:
# Optional: save to file
out_ttl = HERE / "output_characterization_step.ttl"
out_ttl.write_text(g.serialize(format="turtle"))
print(f"Written to {out_ttl}")

Written to /root/semantic-dataspace/semantic-schemas/schemas/characterization/step/PMDCo/docs/output_characterization_step.ttl


## Step 4: Validate against the SHACL shape

SHACL (Shapes Constraint Language) defines rules that a valid RDF graph must
satisfy. The shape file `specs/shape.ttl` checks that:

- The characterisation step has exactly one label.
- Every input is recorded as an IRI (not a plain text string).
- Every test condition has a label, and its value (if given) is a number.

In [7]:
shapes_graph = rdflib.Graph().parse(str(SCHEMA / "specs" / "shape.ttl"))

conforms, results_graph, _ = pyshacl.validate(
    g,
    shacl_graph = shapes_graph,
)

print(f"Conforms: {conforms}")
if not conforms:
    SH = rdflib.Namespace("http://www.w3.org/ns/shacl#")
    for result in results_graph.subjects(rdflib.RDF.type, SH.ValidationResult):
        msg  = results_graph.value(result, SH.resultMessage)
        path = results_graph.value(result, SH.resultPath)
        loc  = str(path).rsplit("#", 1)[-1].rsplit("/", 1)[-1] if path else None
        print(f"\n  x {msg}" + (f"\n    property: {loc}" if loc else ""))

Conforms: True


## Step 5: Inspect the graph

The cells below use SPARQL (a query language for RDF graphs) to confirm that
the key facts from your input were recorded correctly.

You do not need to understand SPARQL to use this notebook. Just run the cells
and check that the output matches what you entered.

In [8]:
# Flatten the Dataset into a plain Graph for SPARQL queries
flat = rdflib.Graph()
for s, p, o, _ in g.quads():
    flat.add((s, p, o))

OBI = rdflib.Namespace("http://purl.obolibrary.org/obo/OBI_")

# Print step label
step_iri = next(flat.subjects(rdflib.RDF.type, OBI["0000070"]))
label    = flat.value(step_iri, rdflib.RDFS.label)
print(f"Characterisation step : {step_iri}")
print(f"Label                 : {label}")

Characterisation step : https://w3id.org/pmd/co/test/char-step-tensile-test-specimen-1
Label                 : Tensile test specimen 1


In [9]:
SPARQL_IO = """
PREFIX obi:  <http://purl.obolibrary.org/obo/OBI_>
PREFIX bfo:  <http://purl.obolibrary.org/obo/BFO_>

SELECT ?role ?iri
WHERE {
  { ?step a obi:0000070 ; obi:0000293 ?iri . BIND("input"       AS ?role) }
  UNION
  { ?step a obi:0000070 ; obi:0000299 ?iri . BIND("output"      AS ?role) }
  UNION
  { ?step a obi:0000070 ; bfo:0000062 ?iri . BIND("preceded by" AS ?role) }
}
ORDER BY ?role ?iri
"""

rows = list(flat.query(SPARQL_IO))
if rows:
    print(f"{'Role':<14}  IRI")
    print("-" * 70)
    for r in rows:
        print(f"{str(r.role):<14}  {r.iri}")
else:
    print("No inputs, outputs, or predecessors recorded.")

Role            IRI
----------------------------------------------------------------------
input           https://example.org/specimens/316L-tensile-bar-1


In [10]:
SPARQL_COND = """
PREFIX obi:  <http://purl.obolibrary.org/obo/OBI_>
PREFIX pmd:  <https://w3id.org/pmd/co/>
PREFIX rdfs: <http://www.w3.org/2000/01/rdf-schema#>
PREFIX qudt: <http://qudt.org/schema/qudt/>

SELECT ?name ?value ?unit
WHERE {
  ?step a obi:0000070 ;
        pmd:PMD_0000016 ?cond .
  ?cond rdfs:label ?name .
  OPTIONAL { ?cond qudt:value ?value . }
  OPTIONAL { ?cond qudt:hasUnit ?unit . }
}
ORDER BY ?name
"""

rows = list(flat.query(SPARQL_COND))
if rows:
    print(f"{'Test Condition':<30}  {'Value':>10}  Unit")
    print("-" * 60)
    for r in rows:
        val  = f"{float(r.value):>10.4g}" if r.value else f"{'n/a':>10}"
        unit = str(r.unit) if r.unit else ""
        print(f"{str(r.name):<30}  {val}  {unit}")
else:
    print("No test conditions recorded.")

Test Condition                       Value  Unit
------------------------------------------------------------
Strain Rate                        0.00025  1/s
Temperature                             23  °C
Test Standard                          n/a  ISO 6892-1


## Summary

| Step | What happens |
|---|---|
| 1 | You fill in `example.input.json` with the test name, specimen, and conditions |
| 2 | The data is converted to an OO-LD document (ontology-mapped JSON) |
| 3 | The OO-LD is parsed into an RDF graph |
| 4 | The graph is validated against the SHACL shape |
| 5 | Key facts are extracted from the graph to confirm correctness |

To describe a different characterisation, edit `docs/example.input.json` and re-run all cells.


## Further reading

- [OO-LD primer](../../../docs/oold-primer.md): what OO-LD is and how it works
- [Schema format reference](../../../docs/schema-format.md): how to write your own schema
- [Schema patterns](../../../docs/schema-patterns.md): inheritance and composition patterns
- `characterization/tensile-test/TTO/` is a specialised schema for ISO 6892-1 tensile tests
  (see its own notebook for a richer demo including measured properties)
- See `workflow/PMDCo/docs/3_workflow_cross_domain.ipynb` for a cross-domain demo using this schema